# Detecção de Neoplasias Mamárias com Redes Neurais Convolucionais

Este notebook implementa um pipeline para a classificação de neoplasias mamárias em imagens de mamografia utilizando Redes Neurais Convolucionais (ResNet50) e técnicas de balanceamento e redução de dimensionalidade. O objetivo é auxiliar o diagnóstico clínico precoce do câncer de mama.

## 📌 Visão Geral

O projeto investiga três abordagens:
1.  **ResNet50 (baseline)**: Classificação direta das imagens.
2.  **SMOTE + PCA + Random Forest + ResNet50**: Combina SMOTE para balanceamento, PCA para redução de dimensionalidade e Random Forest como classificador auxiliar, com features extraídas da ResNet50.
3.  **SMOTEENN + Random Forest + ResNet50**: Similar ao anterior, mas utilizando SMOTEENN para balanceamento e limpeza de ruído.

## 🗂️ Estrutura do Repositório

```
breast-cancer-cnn/
│
├── notebook/
│   └── breast_cancer_classification.ipynb   # Notebook principal com todo o pipeline
│
├── data/
│   ├── mass_case_description_train_set.csv
│   ├── calc_case_description_train_set.csv
│   ├── mass_case_description_test_set.csv
│   └── calc_case_description_test_set.csv
│
└── README.md
```

## 📊 Dataset

**CBIS-DDSM** — *Curated Breast Imaging Subset of Digital Database for Screening Mammography*  
Fonte: [The Cancer Imaging Archive (TCIA)](https://www.cancerimagingarchive.net/collection/cbis-ddsm/)

Este notebook irá baixar e processar as imagens DICOM e os metadados CSV desta coleção.

## ⚙️ Metodologia

O pipeline será desenvolvido em etapas progressivas, focando no balanceamento de classes e na avaliação dos modelos.

---

## 🚀 Implementação

### 1. Instalação e Importação de Bibliotecas

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm

# Para processamento de imagens DICOM
import pydicom
from pydicom.pixel_data_handlers.util import apply_voi_lut

# Para modelos de Deep Learning
import tensorflow as tf
from tensorflow.keras.applications.resnet50 import ResNet50, preprocess_input
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D
from tensorflow.keras.models import Model
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# Para balanceamento e redução de dimensionalidade
from imblearn.over_sampling import SMOTE, SMOTEENN
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier

# Para métricas de avaliação
from sklearn.metrics import confusion_matrix, roc_curve, auc, precision_recall_curve, average_precision_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import label_binarize

# Para download do TCIA
from tcia_utils import nbia

print("Bibliotecas importadas com sucesso!")

### 2. Download e Carregamento dos Metadados

Os arquivos CSV de metadados foram baixados previamente e estão localizados na pasta `data/`.


In [ ]:
# Caminho para os arquivos CSV
DATA_DIR = os.path.join("..")

mass_train_df = pd.read_csv(os.path.join(DATA_DIR, "data", "mass_case_description_train_set.csv"))
calc_train_df = pd.read_csv(os.path.join(DATA_DIR, "data", "calc_case_description_train_set.csv"))
mass_test_df = pd.read_csv(os.path.join(DATA_DIR, "data", "mass_case_description_test_set.csv"))
calc_test_df = pd.read_csv(os.path.join(DATA_DIR, "data", "calc_case_description_test_set.csv"))

print("Metadados carregados com sucesso!")

# Exibir as primeiras linhas dos dataframes para verificação
print("\nMass Train Data (primeiras 2 linhas):")
print(mass_train_df.head(2))

print("\nCalc Train Data (primeiras 2 linhas):")
print(calc_train_df.head(2))

print("\nMass Test Data (primeiras 2 linhas):")
print(mass_test_df.head(2))

print("\nCalc Test Data (primeiras 2 linhas):")
print(calc_test_df.head(2))

### 3. Pré-processamento dos Metadados

Unificar os dataframes de massa e calcificação, e criar uma coluna de label binária (0 para BENIGN/BENIGN_WITHOUT_CALLBACK, 1 para MALIGNANT).


In [ ]:
# Adicionar coluna \'Type\' para diferenciar massas de calcificações
mass_train_df["Type"] = "MASS"
calc_train_df["Type"] = "CALC"
mass_test_df["Type"] = "MASS"
calc_test_df["Type"] = "CALC"

# Renomear colunas para consistência (breast_density vs breast density)
calc_train_df.rename(columns={"breast density": "breast_density"}, inplace=True)
calc_test_df.rename(columns={"breast density": "breast_density"}, inplace=True)

# Unificar dataframes de treino e teste
train_df = pd.concat([mass_train_df, calc_train_df], ignore_index=True)
test_df = pd.concat([mass_test_df, calc_test_df], ignore_index=True)

# Criar coluna de label binária
# 0: BENIGN / BENIGN_WITHOUT_CALLBACK
# 1: MALIGNANT

def create_label(pathology):
    if "MALIGNANT" in pathology.upper():
        return 1
    elif "BENIGN" in pathology.upper():
        return 0
    return -1 # Para casos desconhecidos, que devem ser tratados ou removidos

train_df["label"] = train_df["pathology"].apply(create_label)
test_df["label"] = test_df["pathology"].apply(create_label)

# Remover linhas com labels -1 (se houver)
train_df = train_df[train_df["label"] != -1].reset_index(drop=True)
test_df = test_df[test_df["label"] != -1].reset_index(drop=True)

print("Pré-processamento de metadados concluído.")
print("Distribuição de labels no conjunto de treino:")
print(train_df["label"].value_counts())
print("\nDistribuição de labels no conjunto de teste:")
print(test_df["label"].value_counts())

### 4. Download das Imagens DICOM

Utilizaremos a biblioteca `tcia_utils` para baixar as imagens DICOM. O `image file path` nos metadados CSV contém o `SeriesInstanceUID` necessário para o download.

**Nota:** O download de todas as imagens pode levar um tempo considerável e consumir bastante espaço em disco. Para fins de demonstração, podemos limitar o número de imagens a serem baixadas ou processar apenas um subconjunto.


In [ ]:
# Função para extrair SeriesInstanceUID do \'image file path\'
def extract_series_uid(image_file_path):
    # O SeriesInstanceUID é a última parte do caminho antes do .dcm (se houver)
    # Ex: Mass-Training_P_00001_LEFT_CC/1.3.6.1.4.1.9590.100.1.2.117041576511324414842508325652101471266.dcm
    parts = image_file_path.split("/")
    if len(parts) > 1:
        uid_with_dcm = parts[-1]
        return uid_with_dcm.split(".")[0] # Remove .dcm se existir
    return None

# Aplicar a função para extrair SeriesInstanceUID
train_df["SeriesInstanceUID"] = train_df["image file path"].apply(extract_series_uid)
test_df["SeriesInstanceUID"] = test_df["image file path"].apply(extract_series_uid)

# Remover duplicatas de SeriesInstanceUID para evitar downloads redundantes
train_series_uids = train_df["SeriesInstanceUID"].dropna().unique().tolist()
test_series_uids = test_df["SeriesInstanceUID"].dropna().unique().tolist()

all_series_uids = list(set(train_series_uids + test_series_uids))

print(f"Total de SeriesInstanceUIDs únicos para download: {len(all_series_uids)}")

# Diretório para salvar as imagens DICOM
DICOM_DIR = os.path.join("..", "dicom_images")
os.makedirs(DICOM_DIR, exist_ok=True)

# Função para baixar séries DICOM
def download_dicom_series(series_uids, download_path):
    print(f"Iniciando download de {len(series_uids)} séries DICOM...")
    series_data_list = [{'SeriesInstanceUID': uid} for uid in series_uids]
    nbia.downloadSeries(series_data=series_data_list, path=download_path)
    print("Download de séries DICOM concluído.")

# **ATENÇÃO**: O download de todas as imagens pode ser muito grande e demorado.
# Para testar, você pode limitar o número de UIDs a serem baixados.
# Exemplo: download_dicom_series(all_series_uids[:10], DICOM_DIR)

# Descomente a linha abaixo para realizar o download completo (pode levar horas/dias e GBs de dados)
download_dicom_series(all_series_uids[:3], DICOM_DIR) # Baixar 3 séries para demonstração

# Para fins de demonstração, vamos simular o download ou usar um subconjunto muito pequeno
print("\nSimulando download de imagens DICOM. Para download real, descomente a linha `download_dicom_series(all_series_uids, DICOM_DIR)`.")


### 5. Pré-processamento das Imagens DICOM

Esta seção será implementada após o download das imagens. Incluirá:
- Leitura de arquivos DICOM.
- Conversão para arrays NumPy.
- Aplicação de VOI LUT (Value of Interest Look Up Table) para correção de brilho/contraste.
- Normalização e redimensionamento das imagens.
- Salvamento das imagens pré-processadas em um formato mais comum (e.g., PNG/JPG) para facilitar o carregamento.


In [ ]:
# Placeholder para a função de pré-processamento de imagens
def load_and_preprocess_dicom(dicom_path, target_size=(224, 224)):
    try:
        dicom = pydicom.dcmread(dicom_path)
        data = dicom.pixel_array

        # Aplicar VOI LUT para correção de brilho/contraste
        if \'VOILUTFunction\' in dicom and dicom.VOILUTFunction == \'SIGMOID\':
            data = apply_voi_lut(data, dicom, preferred_voi_lut_func=\'SIGMOID\')
        else:
            data = apply_voi_lut(data, dicom)

        # Normalizar para 0-255 (se não estiver já)
        data = data - np.min(data)
        if np.max(data) != 0:
            data = data / np.max(data)
        data = (data * 255).astype(np.uint8)

        # Redimensionar
        image = tf.image.resize(np.expand_dims(data, axis=-1), target_size).numpy()
        return image
    except Exception as e:
        print(f"Erro ao processar DICOM {dicom_path}: {e}")
        return None

print("Função `load_and_preprocess_dicom` definida.")

### 6. Extração de Features com ResNet50 Pré-treinada

Utilizaremos a ResNet50 pré-treinada no ImageNet como extrator de features. A camada `GlobalAveragePooling2D` será usada para obter um vetor de features de cada imagem.


In [ ]:
# Carregar modelo ResNet50 pré-treinado (sem as camadas finais de classificação)
base_model = ResNet50(weights=\'imagenet\', include_top=False, input_shape=(224, 224, 3))
x = base_model.output
x = GlobalAveragePooling2D()(x)
feature_extractor = Model(inputs=base_model.input, outputs=x)

print("Extrator de features ResNet50 carregado.")

### 5.1. Carregamento e Pré-processamento das Imagens DICOM (Subconjunto para Demonstração)

Para fins de demonstração e para evitar downloads muito longos, vamos processar um subconjunto pequeno das imagens. Em um cenário real, todas as imagens seriam baixadas e processadas.


In [ ]:
# Função auxiliar para encontrar o caminho completo do arquivo DICOM
def find_dicom_file(series_uid, dicom_base_dir):
    for root, _, files in os.walk(dicom_base_dir):
        for file in files:
            if series_uid in file and file.endswith(".dcm"):
                return os.path.join(root, file)
    return None

# Definir um diretório para salvar as imagens pré-processadas (se necessário)
PREPROCESSED_IMAGES_DIR = os.path.join("..", "preprocessed_images")
os.makedirs(PREPROCESSED_IMAGES_DIR, exist_ok=True)

# Listas para armazenar features e labels
X_train_features = []
y_train = []
X_test_features = []
y_test = []

# Processar um subconjunto pequeno para demonstração
# Limitar a 100 imagens de treino e 50 de teste para agilizar

print("\nProcessando imagens de treino (subconjunto)...")
processed_train_uids = set()
for index, row in tqdm(train_df.iterrows(), total=len(train_df)):
    if len(processed_train_uids) >= 100: # Limite para demonstração
        break
    series_uid = row["SeriesInstanceUID"]
    label = row["label"]
    if series_uid and series_uid not in processed_train_uids:
        dicom_path = find_dicom_file(series_uid, DICOM_DIR)
        if dicom_path:
            image = load_and_preprocess_dicom(dicom_path)
            if image is not None:
                # ResNet50 espera 3 canais, então replicamos o canal único
                image_3_channels = np.stack([image[:,:,0], image[:,:,0], image[:,:,0]], axis=-1)
                image_3_channels = preprocess_input(image_3_channels) # Pré-processamento específico da ResNet50
                feature = feature_extractor.predict(np.expand_dims(image_3_channels, axis=0))
                X_train_features.append(feature.flatten())
                y_train.append(label)
                processed_train_uids.add(series_uid)

print("\nProcessando imagens de teste (subconjunto)...")
processed_test_uids = set()
for index, row in tqdm(test_df.iterrows(), total=len(test_df)):
    if len(processed_test_uids) >= 50: # Limite para demonstração
        break
    series_uid = row["SeriesInstanceUID"]
    label = row["label"]
    if series_uid and series_uid not in processed_test_uids:
        dicom_path = find_dicom_file(series_uid, DICOM_DIR)
        if dicom_path:
            image = load_and_preprocess_dicom(dicom_path)
            if image is not None:
                image_3_channels = np.stack([image[:,:,0], image[:,:,0], image[:,:,0]], axis=-1)
                image_3_channels = preprocess_input(image_3_channels)
                feature = feature_extractor.predict(np.expand_dims(image_3_channels, axis=0))
                X_test_features.append(feature.flatten())
                y_test.append(label)
                processed_test_uids.add(series_uid)

X_train_features = np.array(X_train_features)
y_train = np.array(y_train)
X_test_features = np.array(X_test_features)
y_test = np.array(y_test)

print(f"Shape das features de treino: {X_train_features.shape}")
print(f"Shape das labels de treino: {y_train.shape}")
print(f"Shape das features de teste: {X_test_features.shape}")
print(f"Shape das labels de teste: {y_test.shape}")

### 8. Treinamento e Avaliação dos Modelos

Agora que temos as features extraídas, podemos treinar e avaliar os três modelos propostos.

#### 8.1. Modelo 1: ResNet50 (Baseline)

Para o modelo baseline, usaremos as features extraídas da ResNet50 diretamente para treinar um classificador simples (e.g., Logistic Regression ou um pequeno MLP). Para manter a simplicidade e focar na comparação das técnicas de balanceamento, vamos usar um RandomForestClassifier aqui também, mas sem SMOTE/PCA.


In [ ]:
print("\n--- Treinando Modelo 1: ResNet50 (Baseline) ---")
model1 = RandomForestClassifier(random_state=42)
model1.fit(X_train_features, y_train)
y_pred_model1 = model1.predict(X_test_features)
y_prob_model1 = model1.predict_proba(X_test_features)[:, 1]

print("\nMatriz de Confusão - Modelo 1:")
cm1 = confusion_matrix(y_test, y_pred_model1)
print(cm1)

# Curva ROC e AUC
fpr1, tpr1, _ = roc_curve(y_test, y_prob_model1)
auc_roc1 = auc(fpr1, tpr1)
print(f"AUC-ROC - Modelo 1: {auc_roc1:.2f}")

# Curva Precision-Recall e AP
precision1, recall1, _ = precision_recall_curve(y_test, y_prob_model1)
ap_score1 = average_precision_score(y_test, y_prob_model1)
print(f"AP Score - Modelo 1: {ap_score1:.2f}")

#### 8.2. Modelo 2: SMOTE + PCA + Random Forest + ResNet50


In [ ]:
print("\n--- Treinando Modelo 2: SMOTE + PCA + Random Forest + ResNet50 ---")

# Aplicar SMOTE
sm = SMOTE(random_state=42)
X_train_smote, y_train_smote = sm.fit_resample(X_train_features, y_train)
print(f"Shape das features de treino após SMOTE: {X_train_smote.shape}")
print(f"Distribuição de labels após SMOTE: {pd.Series(y_train_smote).value_counts()}")

# Aplicar PCA
pca = PCA(n_components=0.95, random_state=42) # Manter 95% da variância
X_train_pca = pca.fit_transform(X_train_smote)
X_test_pca = pca.transform(X_test_features)
print(f"Shape das features de treino após PCA: {X_train_pca.shape}")
print(f"Shape das features de teste após PCA: {X_test_pca.shape}")

# Treinar Random Forest
model2 = RandomForestClassifier(random_state=42)
model2.fit(X_train_pca, y_train_smote)
y_pred_model2 = model2.predict(X_test_pca)
y_prob_model2 = model2.predict_proba(X_test_pca)[:, 1]

print("\nMatriz de Confusão - Modelo 2:")
cm2 = confusion_matrix(y_test, y_pred_model2)
print(cm2)

# Curva ROC e AUC
fpr2, tpr2, _ = roc_curve(y_test, y_prob_model2)
auc_roc2 = auc(fpr2, tpr2)
print(f"AUC-ROC - Modelo 2: {auc_roc2:.2f}")

# Curva Precision-Recall e AP
precision2, recall2, _ = precision_recall_curve(y_test, y_prob_model2)
ap_score2 = average_precision_score(y_test, y_prob_model2)
print(f"AP Score - Modelo 2: {ap_score2:.2f}")

#### 8.3. Modelo 3: SMOTEENN + Random Forest + ResNet50


In [ ]:
print("\n--- Treinando Modelo 3: SMOTEENN + Random Forest + ResNet50 ---")

# Aplicar SMOTEENN
sme = SMOTEENN(random_state=42)
X_train_smoteenn, y_train_smoteenn = sme.fit_resample(X_train_features, y_train)
print(f"Shape das features de treino após SMOTEENN: {X_train_smoteenn.shape}")
print(f"Distribuição de labels após SMOTEENN: {pd.Series(y_train_smoteenn).value_counts()}")

# Treinar Random Forest (sem PCA para este modelo, como no original)
model3 = RandomForestClassifier(random_state=42)
model3.fit(X_train_smoteenn, y_train_smoteenn)
y_pred_model3 = model3.predict(X_test_features)
y_prob_model3 = model3.predict_proba(X_test_features)[:, 1]

print("\nMatriz de Confusão - Modelo 3:")
cm3 = confusion_matrix(y_test, y_pred_model3)
print(cm3)

# Curva ROC e AUC
fpr3, tpr3, _ = roc_curve(y_test, y_prob_model3)
auc_roc3 = auc(fpr3, tpr3)
print(f"AUC-ROC - Modelo 3: {auc_roc3:.2f}")

# Curva Precision-Recall e AP
precision3, recall3, _ = precision_recall_curve(y_test, y_prob_model3)
ap_score3 = average_precision_score(y_test, y_prob_model3)
print(f"AP Score - Modelo 3: {ap_score3:.2f}")

### 9. Visualização dos Resultados

#### 9.1. Matrizes de Confusão


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle("Matrizes de Confusão dos Modelos")

sns.heatmap(cm1, annot=True, fmt="d", cmap="Blues", ax=axes[0])
axes[0].set_title("Modelo 1: ResNet50 (Baseline)")
axes[0].set_xlabel("Previsto")
axes[0].set_ylabel("Real")

sns.heatmap(cm2, annot=True, fmt="d", cmap="Blues", ax=axes[1])
axes[1].set_title("Modelo 2: SMOTE + PCA + RF + ResNet50")
axes[1].set_xlabel("Previsto")
axes[1].set_ylabel("Real")

sns.heatmap(cm3, annot=True, fmt="d", cmap="Blues", ax=axes[2])
axes[2].set_title("Modelo 3: SMOTEENN + RF + ResNet50")
axes[2].set_xlabel("Previsto")
axes[2].set_ylabel("Real")

plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.savefig("../results/confusion_matrices.png")
plt.show()

#### 9.2. Curvas ROC


In [ ]:
plt.figure(figsize=(8, 6))
plt.plot(fpr1, tpr1, label=f"Modelo 1 (AUC = {auc_roc1:.2f})")
plt.plot(fpr2, tpr2, label=f"Modelo 2 (AUC = {auc_roc2:.2f})")
plt.plot(fpr3, tpr3, label=f"Modelo 3 (AUC = {auc_roc3:.2f})")
plt.plot([0, 1], [0, 1], "k--", label="Aleatório")
plt.xlabel("Taxa de Falsos Positivos (FPR)")
plt.ylabel("Taxa de Verdadeiros Positivos (TPR)")
plt.title("Curvas ROC")
plt.legend()
plt.grid(True)
plt.savefig("../results/roc_curves.png")
plt.show()

#### 9.3. Curvas Precision-Recall


In [ ]:
plt.figure(figsize=(8, 6))
plt.plot(recall1, precision1, label=f"Modelo 1 (AP = {ap_score1:.2f})")
plt.plot(recall2, precision2, label=f"Modelo 2 (AP = {ap_score2:.2f})")
plt.plot(recall3, precision3, label=f"Modelo 3 (AP = {ap_score3:.2f})")
plt.xlabel("Recall")
plt.ylabel("Precisão")
plt.title("Curvas Precision-Recall")
plt.legend()
plt.grid(True)
plt.savefig("../results/precision_recall_curves.png")
plt.show()

### 10. Tabela de Resultados


In [ ]:
results_data = {
    "Modelo": [
        "ResNet50 (baseline)",
        "SMOTE + PCA + RF + ResNet50",
        "SMOTEENN + RF + ResNet50",
    ],
    "AUC-ROC": [auc_roc1, auc_roc2, auc_roc3],
    "AP Score": [ap_score1, ap_score2, ap_score3],
}

results_df = pd.DataFrame(results_data)
print("\n--- Tabela de Resultados ---")
print(results_df.to_markdown(index=False))

# Salvar a tabela de resultados em um arquivo Markdown
results_df.to_markdown("../results/results_table.md", index=False)

print("\nNotebook concluído. Verifique a pasta \"results\" para as imagens e a tabela de resultados.")